# NR-PANNs-MR — Jupyter 3 nhánh (bản sửa toàn bộ)

Wavegram (64) + Log-Mel (64) + Multi-Resolution Transient (32) → NRE-V4 (3 reliability scores) → gated attention fusion (160→128) → pretrained PANNs Conv2–6/FC1 → ESC-50 (50 classes).

**Chạy bằng Kernel → Restart Kernel and Run All** sau khi sửa `ROOT`. Các file `mr_train.py`, `mr_model.py`, `mr_encoder.py` phải nằm cùng thư mục với notebook. Chạy mới từ pretrained AudioSet, không tự resume. Kết quả trong `runs/nrpanns_mr_fixed`, không ghi đè model 2 nhánh.

## 1. Phụ thuộc (chỉ cần cài một lần)

In [ ]:
%pip install -q numpy scipy soundfile pandas matplotlib torchlibrosa librosa h5py

## 2. Import an toàn và kiểm tra đúng mô hình 3 nhánh

In [ ]:
import sys, json, csv, inspect, importlib
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Notebook phải được mở trong thư mục có 3 file source phiên bản FIXED.
SOURCE_DIR = Path.cwd().resolve()
needed = [SOURCE_DIR / f for f in ('mr_encoder.py', 'mr_model.py', 'mr_train.py')]
if not all(p.is_file() for p in needed):
    raise FileNotFoundError(f'Chạy Jupyter từ thư mục chứa notebook + mr_*.py: {SOURCE_DIR}; thiếu {[str(p) for p in needed if not p.is_file()]}')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
# Không dùng module tên train/nr_panns dễ va chạm với notebook 2 nhánh.
for module_name in ('mr_train', 'mr_model', 'mr_encoder'):
    existing = sys.modules.get(module_name)
    if existing is not None and Path(getattr(existing, '__file__', '')).resolve().parent != SOURCE_DIR:
        del sys.modules[module_name]
import mr_train, mr_model, mr_encoder
importlib.reload(mr_train)
from mr_train import train_fold, seed_everything, loaders_for, build_model, gradient_diagnostics
from mr_model import NRPANNsMR
from mr_encoder import DEFAULT_CHECKPOINT_NAME
assert Path(inspect.getfile(build_model)).resolve() == (SOURCE_DIR / 'mr_train.py').resolve()
assert Path(inspect.getfile(NRPANNsMR)).resolve() == (SOURCE_DIR / 'mr_model.py').resolve()
print('Train file:', inspect.getfile(build_model))
print('Model class:', NRPANNsMR)
print('forward_features:', inspect.signature(NRPANNsMR.forward_features))
assert 'return_reliability' in inspect.signature(NRPANNsMR.forward_features).parameters


## 3. GPU: kiểm tra phép tính thực tế, không chỉ `cuda.is_available()`

In [ ]:
DEVICE = torch.device('cpu')
print('Torch:', torch.__version__, '| CUDA runtime:', torch.version.cuda)
if torch.cuda.is_available():
    try:
        print('GPU:', torch.cuda.get_device_name(0))
        test = torch.ones((32, 32), device='cuda:0')
        value = float((test @ test).mean().item())
        torch.cuda.synchronize()
        del test
        DEVICE = torch.device('cuda:0')
        print('CUDA computation OK:', value)
    except Exception as exc:
        print('CUDA unavailable despite detection:', repr(exc))
        print('GPU error: restart kernel / inspect nvidia-smi before GPU training.')
print('Selected device:', DEVICE)
# Với GPU bận, KHÔNG chuyển model sang cuda cho đến khi phép tính trên thành công.


## 4. Cấu hình (chỉnh ROOT cho đúng máy)

In [ ]:
ROOT = Path('/home/robinson/robotic/PANNS')
CHECKPOINT = ROOT / 'checkpoints' / DEFAULT_CHECKPOINT_NAME
NRE_CHECKPOINT = None  # checkpoint NRE-V3 không tương thích NRE-V4
OUTPUT = SOURCE_DIR / 'runs' / 'nrpanns_mr_fixed'
TEST_FOLD = 5
EPOCHS = 100
PATIENCE = 20
BATCH_SIZE = 2  # RTX 4060 Laptop 8GB: bắt đầu với 2, tăng 4 nếu đủ VRAM
WORKERS = 0
LR_BACKBONE = 2e-5
LR_NEW = 1e-4
WEIGHT_DECAY = 1e-4
CLIP = 1.0
ALPHA_INIT = 0.1
SEED = 42
USE_AMP = False
args = SimpleNamespace(root=ROOT, checkpoint=CHECKPOINT, nre_checkpoint=NRE_CHECKPOINT,
    output=OUTPUT, epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE,
    workers=WORKERS, lr_backbone=LR_BACKBONE, lr_new=LR_NEW,
    weight_decay=WEIGHT_DECAY, clip=CLIP, alpha=ALPHA_INIT, seed=SEED, amp=USE_AMP)
print('Test fold:', TEST_FOLD, '| Val fold:', TEST_FOLD % 5 + 1, '| Output:', OUTPUT)


## 5. Kiểm tra dữ liệu (không dùng pin_memory)

In [ ]:
required = [ROOT / 'audioset_tagging_cnn' / 'pytorch' / 'models.py', CHECKPOINT,
            ROOT / 'data' / 'ESC-50' / 'meta' / 'esc50.csv', ROOT / 'data' / 'ESC-50' / 'audio']
for p in required:
    print('OK     ' if p.exists() else 'MISSING', p)
if not all(p.exists() for p in required):
    raise FileNotFoundError('Thiếu file/thư mục, kiểm tra ROOT và CHECKPOINT')
train_loader, val_loader, test_loader, val_fold = loaders_for(
    ROOT / 'data' / 'ESC-50', TEST_FOLD, BATCH_SIZE, WORKERS, SEED + TEST_FOLD)
assert not train_loader.pin_memory and not val_loader.pin_memory and not test_loader.pin_memory
print('Train/Val/Test:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
print('Train folds:', sorted(set(range(1, 6)) - {TEST_FOLD, val_fold}))
batch = next(iter(train_loader))
print('Waveforms:', tuple(batch['waveform'].shape), 'Labels:', tuple(batch['label'].shape))
assert bool(torch.isfinite(batch['waveform']).all())


## 6. Kiểm tra model, 3 reliability scores và shape (không cần train)

In [ ]:
seed_everything(SEED + TEST_FOLD)
model_check = build_model(ROOT, CHECKPOINT, NRE_CHECKPOINT, ALPHA_INIT)
assert isinstance(model_check, NRPANNsMR), f'Wrong model: {type(model_check)}'
print('Correct model:', type(model_check).__name__)
model_check = model_check.to(DEVICE)
model_check.eval()
with torch.inference_mode():
    audio = batch['waveform'][:1].to(DEVICE)
    features, reliability = model_check.forward_features(audio, return_reliability=True)
    logits = model_check.forward_backbone(features)
    logits = model_check.classifier(logits)
    print('Features:', tuple(features.shape), 'Logits:', tuple(logits.shape))
    print('Reliability (w, m, t):', [r.flatten().tolist() for r in reliability])
    assert features.shape[1] == 128 and logits.shape == (1, 50) and len(reliability) == 3
    assert bool(torch.isfinite(logits).all())
print('Trainable parameters:', f'{sum(p.numel() for p in model_check.parameters() if p.requires_grad):,}')
del model_check, audio, features, reliability, logits
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('Forward OK')


## 7. Huấn luyện test Fold 5 (khởi tạo lại pretrained AudioSet từ epoch 1)

In [ ]:
# LƯU Ý: Chạy cell này sẽ train mới và có thể ghi đè best.pt trong OUTPUT/fold_5.
# Nếu muốn giữ lần chạy cũ, sửa OUTPUT/args.output trước khi chạy.
if DEVICE.type == 'cpu':
    raise RuntimeError('Hiện đang dùng CPU. Kiểm tra GPU trước khi bắt đầu full fine-tuning.')
result = train_fold(args, TEST_FOLD, DEVICE)
print('Best epoch:', result['best_epoch'])
print('Best validation:', f"{result['best_val_accuracy']:.2%}")
print('Held-out test:', f"{result['test_accuracy']:.2%}")


## 8. Xem kết quả mà không train lại

In [ ]:
history_file = OUTPUT / f'fold_{TEST_FOLD}' / 'history.csv'
result_file = OUTPUT / f'fold_{TEST_FOLD}' / 'result.json'
if history_file.exists():
    history = pd.read_csv(history_file)
    display(history.tail())
    for cols, ylabel in [(['train_accuracy','val_accuracy'], 'Accuracy'), (['train_loss','val_loss'], 'Loss')]:
        ax = history.plot(x='epoch', y=cols, figsize=(9,4))
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        plt.show()
else:
    print('Chưa có history.csv. Hãy train trước.')
if result_file.exists():
    print(json.dumps(json.loads(result_file.read_text()), indent=2))


## 9. Test inference từ best.pt (không train)

In [ ]:
best_path = OUTPUT / f'fold_{TEST_FOLD}' / 'best.pt'
if best_path.exists():
    inference_model = build_model(ROOT, CHECKPOINT, NRE_CHECKPOINT, ALPHA_INIT).to(DEVICE)
    state = torch.load(best_path, map_location='cpu', weights_only=True)
    inference_model.load_state_dict(state['model'], strict=True)
    inference_model.eval()
    sample = next(iter(test_loader))
    with torch.inference_mode():
        pred = inference_model(sample['waveform'][:1].to(DEVICE)).argmax(1).item()
    print('Prediction:', pred, '| Label:', int(sample['label'][0]))
    del inference_model, state
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('Chưa có best.pt; chưa train hoặc đường dẫn OUTPUT không khớp.')


## 10. Lưu ý nghiên cứu

- Test Fold 5 chỉ dùng đánh giá sau khi chọn best checkpoint trên Validation Fold 1.
- NRE-V4 nhận gradient từ cross-entropy; ba output sigmoid **chưa phải reliability được hiệu chuẩn theo SNR**.
- `pin_memory=False` chỉ tránh lỗi pin-memory; nếu tính toán CUDA thật thất bại phải xử lý driver/GPU trước.
- Khi so sánh NR-PANNs 2 nhánh và 3 nhánh, giữ giống fold, seed, pretrained checkpoint, augmentation, training budget.
- Không tự tăng epochs lên 200 nếu validation đã chững lại.